In [ ]:
import os
import requests
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.errors import GraphRecursionError

# TMDB API 키 설정 (환경변수 등에 미리 세팅 필요)
TMDB_API_KEY = os.getenv("TMDB_API_KEY")

@tool
def search_movies(query: str) -> str:
    """영화를 검색한다. 영화 제목을 검색할 수 있다. 예: '인셉션', '기생충', '아이언맨'"""
    response = requests.get(
        "https://api.themoviedb.org/3/search/movie",
        headers={"Authorization": f"Bearer {TMDB_API_KEY}"},
        params={"query": query, "language": "ko-KR"},
    )
    data = response.json()
    results = data.get("results", [])[:5]
    
    if not results:
        return f"'{query}'에 대한 검색 결과가 없습니다."

    output = []
    for movie in results:
        movie_id = movie.get("id")
        title = movie.get("title", "제목 없음")
        year = (movie.get("release_date") or "")[:4]
        rating = movie.get("vote_average", 0)
        overview = movie.get("overview", "줄거리 없음")[:100]
        output.append(f"- [{movie_id}] {title} ({year}) ⭐ {rating}\n  {overview}")
    return "\n".join(output)

@tool
def get_movie_detail(movie_id: int) -> str:
    """영화의 상세정보를 조회한다. movie_id에는 영화의 id가 들어간다."""
    response = requests.get(
        f"https://api.themoviedb.org/3/movie/{movie_id}",
        headers={"Authorization": f"Bearer {TMDB_API_KEY}"},
        params={"language": "ko-KR"},
    )
    data = response.json()
    if not data or data.get("success") is False:
        return "영화가 존재하지 않습니다."
    return str(data)  # Tool의 반환값은 문자열 형태가 안전합니다.

# 에이전트가 사용할 도구 목록
tools = [search_movies, get_movie_detail]